In [1]:
import pandas as pd
import pickle
import os
from nltk.tokenize import sent_tokenize
import nltk
import re
import string
# import gensim
import itertools
import matplotlib.pyplot as plt 
nltk.download('punkt')
import bs4 as bs
import numpy as np
import json
import ast
from functools import reduce
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_score, recall_score

directory = '../data/'
translator = re.compile('[%s]' % re.escape(string.punctuation))
plt.rcParams['figure.figsize'] = (10,7)
plt.style.use('ggplot')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\xtang2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
from openai import OpenAI
client = OpenAI(api_key="sk-proj-fyoBu7GzQG8H6LEq0lPdT3BlbkFJfZqc0U4ZYoIy89R6I2VP")
# client = OpenAI(api_key='sk-None-eGR9kyQrLnY0BzZtfI67T3BlbkFJt8kbEsatG9t8dFMIF4hJ')
# client = OpenAI(api_key="sk-proj-5qzZVEalSawZKz3tQLU1T3BlbkFJW3JlMmi6EKwlqyiNyFT4") # mine

0. processing training and testing sets

In [5]:
# prepare files for different tasks
df_train = pd.read_excel(directory+'output/finetuning/training2.xlsx')
df_test = pd.read_excel(directory+'output/finetuning/testing2.xlsx')
    
df_train_agree = df_train[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]
df_test_agree = df_test[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]

In [6]:
for col in ['agreement_general', 'disagreement_areas']:
    print(df_train_agree[col].value_counts(normalize=True))
    print(df_test_agree[col].value_counts(normalize=True))
    print('\n')

agreement_general
mostly agree           0.696970
disagreement exists    0.229437
irrelevant             0.073593
Name: proportion, dtype: float64
agreement_general
mostly agree           0.724138
disagreement exists    0.172414
irrelevant             0.103448
Name: proportion, dtype: float64


disagreement_areas
Future Policy Direction                                            0.464286
Current Policy Stance                                              0.142857
Monetary Policy Operations                                         0.089286
Monetary Policy Framework                                          0.053571
Current Policy Stance; Future Policy Direction                     0.035714
Exchange Rate Policy                                               0.035714
Economic Assessment; Future Policy Direction                       0.035714
Economic Assessment                                                0.035714
Central Bank Communication                                         0.035714
M

In [67]:
# areas_l = set('; '.join(df_test['disagreement_areas'].value_counts().index).replace('Future Policy Stance', 'Future Policy Direction').split('; '))
# for a in areas_l:
#     df_test_agree['da_'+a] = df_test['disagreement_areas'].apply(lambda x: x==x and a.replace('Future Policy Direction', 'Future Policy Stance') in x)

1. refine the agreement variables using GPT

In [533]:
no predefined values for disagreement + chain of thought
for i, row in df_test_agree.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authority, determine whether the country's authority agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; if the authority mostly agree, leave the "disagreement_areas" field blank. Provide reasoning for your answer. Return a JSON dict without additional texts: \"agreement\", \"disagreement_areas\", \"reason\".''',},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])}
        ],
            model='gpt-4o-2024-08-06', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace('\n',' '))
        df_test_agree.loc[i,'agreement_gpt0'] = result['agreement']
        df_test_agree.loc[i,'disagreement_areas_gpt0'] = result['disagreement_areas']
        df_test_agree.loc[i,'reason_gpt0'] = result['reason']
    except Exception:
        df_test_agree.loc[i, 'error_gpt0'] = chat_completion.choices[0].message.content
df_test_agree['error_gpt0'] = df_test_agree['error_gpt0'].apply(lambda x: json.loads(x.replace('```json','').replace('```','').replace('\n',' ')) if x==x else np.nan)
df_test_agree['agreement_gpt0'] = df_test_agree.apply(lambda x: x['error_gpt0']['agreement'] if x['agreement_gpt0']!=x['agreement_gpt0'] else x['agreement_gpt0'], axis=1)
df_test_agree['disagreement_areas_gpt0'] = df_test_agree.apply(lambda x: x['error_gpt0']['disagreement_areas'] if x['disagreement_areas_gpt0']!=x['disagreement_areas_gpt0'] else x['disagreement_areas_gpt0'], axis=1)
df_test_agree['reason_gpt0'] = df_test_agree.apply(lambda x: x['error_gpt0']['reason'] if x['reason_gpt0']!=x['reason_gpt0'] else x['reason_gpt0'], axis=1)


for i, row in df_train_agree.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authority, determine whether the country's authority agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; if the authority mostly agree, leave the "disagreement_areas" field blank. Provide reasoning for your answer. Return a JSON dict without additional texts: \"agreement\", \"disagreement_areas\", \"reason\".''',},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])}
        ],
            model='gpt-4o-2024-08-06', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace('\n',' '))
        df_train_agree.loc[i,'agreement_gpt0'] = result['agreement']
        df_train_agree.loc[i,'disagreement_areas_gpt0'] = result['disagreement_areas']
        df_train_agree.loc[i,'reason_gpt0'] = result['reason']
    except Exception:
        df_train_agree.loc[i, 'error_gpt0'] = chat_completion.choices[0].message.content
        
df_train_agree['error_gpt0'] = df_train_agree['error_gpt0'].apply(lambda x: json.loads(x.replace('```json','').replace('```','').replace('\n',' ')) if x==x else np.nan)
df_train_agree['agreement_gpt0'] = df_train_agree.apply(lambda x: x['error_gpt0']['agreement'] if x['agreement_gpt0']!=x['agreement_gpt0'] else x['agreement_gpt0'], axis=1)
df_train_agree['disagreement_areas_gpt0'] = df_train_agree.apply(lambda x: x['error_gpt0']['disagreement_areas'] if x['disagreement_areas_gpt0']!=x['disagreement_areas_gpt0'] else x['disagreement_areas_gpt0'], axis=1)
df_train_agree['reason_gpt0'] = df_train_agree.apply(lambda x: x['error_gpt0']['reason'] if x['reason_gpt0']!=x['reason_gpt0'] else x['reason_gpt0'], axis=1)


C:\Users\xtang2\AppData\Local\Temp\ipykernel_17144\676885126.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_agree['error_gpt0'] = df_train_agree['error_gpt0'].apply(lambda x: json.loads(x.replace('```json','').replace('```','').replace('\n',' ')) if x==x else np.nan)
C:\Users\xtang2\AppData\Local\Temp\ipykernel_17144\676885126.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_agree['agreement_gpt0'] = df_train_agree.apply(lambda x: x['error_gpt0']['agreement'] if x['agreement_gpt0

In [467]:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt0']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt0'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt0'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.6724137931034483,
 0.6333554376657824,
 array([[ 8, 10,  0],
        [ 3, 30,  0],
        [ 1,  5,  1]], dtype=int64))

In [513]:
accuracy_score(df_train_agree['agreement_general'], df_train_agree['agreement_gpt0']), f1_score(df_train_agree['agreement_general'], df_train_agree['agreement_gpt0'], average='weighted'), confusion_matrix(df_train_agree['agreement_general'], df_train_agree['agreement_gpt0'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.6233766233766234,
 0.5853750307145014,
 array([[ 42,  60,   0],
        [ 11, 101,   0],
        [  3,  13,   1]], dtype=int64))

In [539]:
# df_test_agree.loc[df_test_agree[df_test_agree['Print ISBN']==9798400258442].index, 'agreement_general'] = 'disagreement exists'
# df_test_agree.at[df_test_agree[df_test_agree['Print ISBN']==9798400258442].index[0], 'disagreement_areas'] = ['Future Policy Direction']
# df_test.loc[df_test[df_test['Print ISBN']==9798400258442].index, 'agreement_general'] = 'disagreement exists'
# df_test.at[df_test[df_test['Print ISBN']==9798400258442].index[0], 'disagreement_areas'] = ['Future Policy Direction']
# df_test.loc[df_test[df_test['Print ISBN']==9798400258442].index, 'buff_stance_future'] = 'loosening bias'

# df_train_agree.loc[df_train_agree[df_train_agree['Print ISBN']==9781484301814].index, 'agreement_general'] = 'disagreement exists'
# df_train_agree.at[df_train_agree[df_train_agree['Print ISBN']==9781484301814].index[0], 'disagreement_areas'] = ['Future Policy Direction']
# df_train.loc[df_train[df_train['Print ISBN']==9781484301814].index, 'agreement_general'] = 'disagreement exists'
# df_train.at[df_train[df_train['Print ISBN']==9781484301814].index[0], 'disagreement_areas'] = ['Future Policy Direction']
# df_train.loc[df_train[df_train['Print ISBN']==9781484301814].index, 'staff_stance_future'] = 'loosening'

# df_train_agree.loc[df_train_agree[df_train_agree['Print ISBN']==9781475551327].index, 'agreement_general'] = 'disagreement exists'
# df_train_agree.at[df_train_agree[df_train_agree['Print ISBN']==9781475551327].index[0], 'disagreement_areas'] = ['Monetary Policy Operations']
# df_train.loc[df_train[df_train['Print ISBN']==9781475551327].index, 'agreement_general'] = 'disagreement exists'
# df_train.at[df_train[df_train['Print ISBN']==9781475551327].index[0], 'disagreement_areas'] = ['Monetary Policy Operations']

# df_train_agree.loc[df_train_agree[df_train_agree['Print ISBN']==9781484317327].index, 'agreement_general'] = 'disagreement exists'
# df_train_agree.at[df_train_agree[df_train_agree['Print ISBN']==9781484317327].index[0], 'disagreement_areas'] = ['Exchange Rate Policy']
# df_train.loc[df_train[df_train['Print ISBN']==9781484317327].index, 'agreement_general'] = 'disagreement exists'
# df_train.at[df_train[df_train['Print ISBN']==9781484317327].index[0], 'disagreement_areas'] = ['Exchange Rate Policy']

# df_train_agree.loc[df_train_agree[df_train_agree['Print ISBN']==9798400243127].index, 'agreement_general'] = 'disagreement exists'
# df_train_agree.at[df_train_agree[df_train_agree['Print ISBN']==9798400243127].index[0], 'disagreement_areas'] = ['Exchange Rate Policy']
# df_train.loc[df_train[df_train['Print ISBN']==9798400243127].index, 'agreement_general'] = 'disagreement exists'
# df_train.at[df_train[df_train['Print ISBN']==9798400243127].index[0], 'disagreement_areas'] = ['Exchange Rate Policy']

# df_train['disagreement_areas'] = df_train['disagreement_areas'].apply(lambda x: '; '.join(x) if type(x)==list else x)
# df_test['disagreement_areas'] = df_test['disagreement_areas'].apply(lambda x: '; '.join(x) if type(x)==list else x)
# df_train_agree['disagreement_areas'] = df_train_agree['disagreement_areas'].apply(lambda x: '; '.join(x) if type(x)==list else x)
# df_test_agree['disagreement_areas'] = df_test_agree['disagreement_areas'].apply(lambda x: '; '.join(x) if type(x)==list else x)

# df_train.to_excel(directory+'output/finetuning/training2.xlsx', index=False)
# df_test.to_excel(directory+'output/finetuning/testing2.xlsx', index=False)


# df_final_orig = pd.read_excel(directory+'output/labelling/labelled/archive/labelled_monetary_v3.xlsx')
# df_train = pd.read_excel(directory+'output/finetuning/training2.xlsx')
# df_test = pd.read_excel(directory+'output/finetuning/testing2.xlsx')
# df_final = pd.concat([df_train,df_test], ignore_index=True)
# key_columns = ['staff_stance_current', 'staff_stance_future', 'buff_stance_current',
#        'buff_stance_future', 'agreement_other', 'disagreement_areas']
# df_final.loc[df_final[df_final['Print ISBN']==9798400258442].index, 'buff_stance_future'] = 'loosening bias'
# for i, row in df_final_orig.iterrows():
#     try:
#         df_temp = df_final[df_final['Print ISBN']==row['Print ISBN']].iloc[0]
#         for col in key_columns:
#             if row[col]!=df_temp[col]:
#                 print(col)
#                 df_final_orig.loc[i, col] = df_temp[col]
#     except Exception:
#         pass
# df_final_orig['disagreement_areas'] = df_final_orig['disagreement_areas'].apply(lambda x: x.replace('Future Policy Stance', 'Future Policy Direction') if x==x else x)
# df_final_orig['agreement_stance_current'] = df_final_orig.apply(lambda x: 'irrelevant' if x['staff_stance_current'] in ['unclear', 'irrelevant'] or x['buff_stance_current'] in ['unclear', 'irrelevant'] else 'mostly agree' if x['staff_stance_current']==x['buff_stance_current'] else 'disagreement exists', axis=1)
# df_final_orig['agreement_stance_future'] = df_final_orig.apply(lambda x: 'irrelevant' if x['staff_stance_future'] in ['unclear', 'irrelevant'] or x['buff_stance_future'] in ['unclear', 'irrelevant'] else 'mostly agree' if x['staff_stance_future']==x['buff_stance_future'] else 'disagreement exists', axis=1)
# df_final_orig.loc[df_final_orig[(df_final_orig['agreement_stance_future']!='disagreement exists')&(df_final_orig['agreement_stance_current']!='disagreement exists')&(df_final_orig['agreement_other']!='disagreement exists')&(df_final_orig['agreement_general']=='disagreement exists')].index, 'agreement_other']='disagreement exists'
# df_final_orig.to_excel(directory+'output/labelling/labelled/labelled_monetary_v3.xlsx', index=False)

2. Zero-shot tests - short/long queries/chain-of-thought

In [111]:
# version 1: simplest
for i, row in df_test_agree.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authority, determine whether the country's authority agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; possible areas include Current Policy Stance, Future Policy Direction, Monetary Policy Framework, Monetary Policy Operations, Central Bank Communication, Institutions, Policy Assessment, Economic Assessment, etc; if the authority mostly agree, leave the "disagreement_areas" field blank. Return a JSON dict without additional texts: \"agreement\", \"disagreement_areas\".''',},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])}
        ],
            model='gpt-4o-2024-08-06', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace('\n',' '))
        df_test_agree.loc[i,'agreement_gpt1'] = result['agreement']
        df_test_agree.loc[i,'disagreement_areas_gpt1'] = result['disagreement_areas']
    except Exception:
        df_test_agree.loc[i, 'error_gpt1'] = chat_completion.choices[0].message.content
        
# df_test_agree['error_gpt1'] = df_test_agree['error_gpt1'].apply(lambda x: json.loads(x.replace('```json','').replace('```','').replace('\n',' ')) if x==x else np.nan)
# df_test_agree['agreement_gpt1'] = df_test_agree.apply(lambda x: x['error_gpt1']['agreement'] if x['agreement_gpt1']!=x['agreement_gpt1'] else x['agreement_gpt1'], axis=1)
# df_test_agree['disagreement_areas_gpt1'] = df_test_agree.apply(lambda x: x['error_gpt1']['disagreement_areas'] if x['disagreement_areas_gpt1']!=x['disagreement_areas_gpt1'] else x['disagreement_areas_gpt1'], axis=1)
# for a in areas_l:
#     df_test_agree['da_'+a+'_gpt1'] = df_test_agree['disagreement_areas_gpt1'].apply(lambda x: x==x and a in x)

In [548]:
# query v1:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt1']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt1'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt1'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.6551724137931034,
 0.6274777853725222,
 array([[ 9, 10,  0],
        [ 5, 28,  0],
        [ 0,  5,  1]], dtype=int64))

In [553]:
df_temp = df_test_agree[(df_test_agree['agreement_general']=='disagreement exists')|(df_test_agree['agreement_gpt1']=='disagreement exists')]
correct_l = []
is_correct_l = []
for i,row in df_temp.iterrows():
    if row['disagreement_areas']!=row['disagreement_areas']:
#         for a in row['disagreement_areas_gpt1']:
#             correct_l.append(False)
#             is_correct_l.append(True)
        pass
    else:
        for a in row['disagreement_areas'].replace('Future Policy Stance', 'Future Policy Direction').split('; '):
            correct_l.append(True)
            is_correct_l.append(a in row['disagreement_areas_gpt1'])
accuracy_score(correct_l, is_correct_l), confusion_matrix(correct_l, is_correct_l, labels=[True, False]) 
# accuracy_score(list(itertools.chain.from_iterable([df_temp['da_'+a] for a in areas_l])), list(itertools.chain.from_iterable([df_temp['da_'+a+'_gpt1'] for a in areas_l]))), precision_score(list(itertools.chain.from_iterable([df_temp['da_'+a] for a in areas_l])), list(itertools.chain.from_iterable([df_temp['da_'+a+'_gpt1'] for a in areas_l])), average='weighted'), confusion_matrix(list(itertools.chain.from_iterable([df_temp['da_'+a] for a in areas_l])), list(itertools.chain.from_iterable([df_temp['da_'+a+'_gpt1'] for a in areas_l])), labels=[True, False])

(0.4782608695652174,
 array([[11, 12],
        [ 0,  0]], dtype=int64))

In [116]:
# version 2: adding brief definitions
for i, row in df_test_agree.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authority, determine whether the country's authority agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; if the authority mostly agree, leave the "disagreement_areas" field blank. Possible areas include:\nCurrent Policy Stance: The current or recent monetary policy stance to influencing the economy through interest rates, money supply, etc.\nFuture Policy Direction: Planned or recommended direction of change in monetary policy stance.\nMonetary Policy Framework: The overall strategy and guidelines governing monetary policy decisions.\nMonetary Policy Operations: The specific actions taken to implement monetary policy, such as open market operations.\nCentral Bank Communication: How the central bank communicates its policy intentions and decisions to the public.\nInstitutions: The structure and role of institutions involved in monetary policy formulation and execution, including the independence of the central bank.\nPolicy Assessment: Evaluation of the effectiveness and outcomes of current or recent monetary policy.\nEconomic Assessment: The analysis of current economic conditions and forecasts that inform monetary policy.\nReturn a JSON dict without additional texts: \"agreement\", \"disagreement_areas\".''',},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])}
        ],
            model='gpt-4o-2024-08-06', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace('\n',' '))
        df_test_agree.loc[i,'agreement_gpt2'] = result['agreement']
        df_test_agree.loc[i,'disagreement_areas_gpt2'] = result['disagreement_areas']
    except Exception:
        df_test_agree.loc[i, 'error_gpt2'] = chat_completion.choices[0].message.content
        
# df_test_agree['error_gpt2'] = df_test_agree['error_gpt2'].apply(lambda x: json.loads(x.replace('```json','').replace('```','').replace('\n',' ')) if x==x else np.nan)
# df_test_agree['agreement_gpt2'] = df_test_agree.apply(lambda x: x['error_gpt2']['agreement'] if x['agreement_gpt2']!=x['agreement_gpt2'] else x['agreement_gpt2'], axis=1)
# df_test_agree['disagreement_areas_gpt2'] = df_test_agree.apply(lambda x: x['error_gpt2']['disagreement_areas'] if x['disagreement_areas_gpt2']!=x['disagreement_areas_gpt2'] else x['disagreement_areas_gpt2'], axis=1)


In [554]:
# query v2:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt2']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt2'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt2'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.6551724137931034,
 0.6274777853725222,
 array([[ 9, 10,  0],
        [ 5, 28,  0],
        [ 0,  5,  1]], dtype=int64))

In [555]:
df_temp = df_test_agree[(df_test_agree['agreement_general']=='disagreement exists')|(df_test_agree['agreement_gpt2']=='disagreement exists')]
correct_l = []
is_correct_l = []
for i,row in df_temp.iterrows():
    if row['disagreement_areas']!=row['disagreement_areas']:
#         for a in row['disagreement_areas_gpt2']:
#             correct_l.append(False)
#             is_correct_l.append(True)
        pass
    else:
        for a in row['disagreement_areas'].replace('Future Policy Stance', 'Future Policy Direction').split('; '):
            correct_l.append(True)
            is_correct_l.append(a in row['disagreement_areas_gpt2'])
accuracy_score(correct_l, is_correct_l), confusion_matrix(correct_l, is_correct_l, labels=[True, False]) 


(0.4782608695652174,
 array([[11, 12],
        [ 0,  0]], dtype=int64))

In [153]:
# version 3: chain-of-thought
for i, row in df_test_agree.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authority, determine whether the country's authority agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; possible areas include Current Policy Stance, Future Policy Direction, Monetary Policy Framework, Monetary Policy Operations, Central Bank Communication, Institutions, Policy Assessment, Economic Assessment, etc; if the authority mostly agree, leave the "disagreement_areas" field blank. Provide reasoning for your answer. Return a JSON dict without additional texts: \"agreement\", \"disagreement_areas\", \"reason\".''',},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])}
        ],
            model='gpt-4o-2024-08-06', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace('\n',' '))
        df_test_agree.loc[i,'agreement_gpt3'] = result['agreement']
        df_test_agree.loc[i,'disagreement_areas_gpt3'] = result['disagreement_areas']
        df_test_agree.loc[i,'reason_gpt3'] = result['reason']
    except Exception:
        df_test_agree.loc[i, 'error_gpt3'] = chat_completion.choices[0].message.content
        
# df_test_agree['error_gpt3'] = df_test_agree['error_gpt3'].apply(lambda x: json.loads(x.replace('```json','').replace('```','').replace('\n',' ')) if x==x else np.nan)
# df_test_agree['agreement_gpt3'] = df_test_agree.apply(lambda x: x['error_gpt3']['agreement'] if x['agreement_gpt3']!=x['agreement_gpt3'] else x['agreement_gpt3'], axis=1)
# df_test_agree['disagreement_areas_gpt3'] = df_test_agree.apply(lambda x: x['error_gpt3']['disagreement_areas'] if x['disagreement_areas_gpt3']!=x['disagreement_areas_gpt3'] else x['disagreement_areas_gpt3'], axis=1)
# df_test_agree['reason_gpt3'] = df_test_agree.apply(lambda x: x['error_gpt3']['reason'] if x['reason_gpt3']!=x['reason_gpt3'] else x['reason_gpt3'], axis=1)


In [556]:
# query v3:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt3']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt3'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt3'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.6551724137931034,
 0.6278122283396117,
 array([[ 9, 10,  0],
        [ 5, 28,  0],
        [ 1,  4,  1]], dtype=int64))

In [557]:
df_temp = df_test_agree[(df_test_agree['agreement_general']=='disagreement exists')|(df_test_agree['agreement_gpt3']=='disagreement exists')]
correct_l = []
is_correct_l = []
for i,row in df_temp.iterrows():
    if row['disagreement_areas']!=row['disagreement_areas']:
#         for a in row['disagreement_areas_gpt3']:
#             correct_l.append(False)
#             is_correct_l.append(True)
        pass
    else:
        for a in row['disagreement_areas'].replace('Future Policy Stance', 'Future Policy Direction').split('; '):
            correct_l.append(True)
            is_correct_l.append(a in row['disagreement_areas_gpt3'])
accuracy_score(correct_l, is_correct_l), confusion_matrix(correct_l, is_correct_l, labels=[True, False]) 

(0.4782608695652174,
 array([[11, 12],
        [ 0,  0]], dtype=int64))

3. Few-shot

In [181]:
# version 4: adding a few examples
example_l = [
    '''Example 1:\nCountry: Mauritius; Year: 2015\n\nPart1 - IMF staff:\n44. The monetary policy stance is broadly appropriate given the low-inflation environment. Further excess liquidity absorption should proceed at a measured pace in order to avoid any sharp increases in market interest rates.\n\nPart2 - Authority:\nThe monetary policy pursued by Bank of Mauritius will remain cautiously accommodative to subdue inflation. In the same vein, efforts to gradually reduce excess domestic liquidity will be enhanced to improve the monetary policy transmission mechanism and without harming the overall money market conditions. The authorities welcome the analysis stating that the real effective exchange rate is broadly in line with fundamentals. Efforts to continue preserving this progress will be pursued with a careful monitoring of continued large inflows from the global business corporate (GBC) sector.\n\nAnswer: {"agreement": "mostly agree", "disagreement_areas": ""}.'''.replace('\n\n','\n'),
    '''Example 2:\nCountry: Guyana; Year: 2017\n\nPart1 - IMF staff:\n51. The accommodative monetary policy is appropriate, but should gradually move towards a neutral stance in 2017. Pass-through from the exchange rate and from the VAT reform to inflation should be closely monitored.\n\nPart2 - Authority:\n11. Monetary policy remained broadly accommodative in 2016. The BoG maintained the bank rate at 5 percent and ensured that liquidity levels in the banking system were conducive to facilitate lending to the private sector. According to BoG data, domestic interest rates fell during 2016 and in the first quarter of 2017. The authorities note staff’s recommendation to gradually tighten monetary policy in 2017. However, with benign price pressures and moderate growth, the BoG will continue to closely monitor domestic and international conditions before adjusting its policy stance.\n\nAnswer: {"agreement": "disagreement exists", "disagreement_areas": "Future Policy Stance"}.'''.replace('\n\n','\n'),
    '''Example 3:\nCountry: Libya; Year: 2023\n\nPart1 - IMF staff:\n53. To maintain public confidence in the nominal anchor, the central bank should continue in its efforts to reunify and avoid frequent changes to the currency peg. The reunification of the central bank is a crucial step towards achieving financial stability and fostering private sector development. Keeping the peg unchanged would allow the central bank to protect foreign exchange reserves and maintain macroeconomic stability amid political and security risks. In 2022, Libya’s external position was broadly in line with the fundamentals and desirable policy settings.\n\nPart2 - Authority:\nMonetary developments are largely driven by large swings in net claims on government as the government resorts to (interest free) monetary financing to meet budget shortfalls. Monetary management is further complicated by the parallel government’s borrowings from CBL-Al Bayda and printing bank notes on which the CBL-Tripoli has no control. The CBL has no instruments to control the monetary aggregates. Since the 2013 prohibition on interest, commercial banks have been setting their own internal rates in lending to the private sector. The CBL intends to develop Islamic finance products and has requested Fund TA.\n\nAnswer: {"agreement": "irrelevant", "disagreement_areas": ""}.'''.replace('\n\n','\n'),
]
for i, row in df_test_agree.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authority, determine whether the country's authority agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; possible areas include Current Policy Stance, Future Policy Direction, Monetary Policy Framework, Monetary Policy Operations, Central Bank Communication, Institutions, Policy Assessment, Economic Assessment, etc; if the authority mostly agree, leave the "disagreement_areas" field blank.\n\n%s\n\nReturn a JSON dict without additional texts: \"agreement\", \"disagreement_areas\".'''%('\n\n'.join(example_l)),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])}
        ],
            model='gpt-4o-2024-08-06', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace('\n',' '))
        df_test_agree.loc[i,'agreement_gpt4'] = result['agreement']
        df_test_agree.loc[i,'disagreement_areas_gpt4'] = result['disagreement_areas']
    except Exception:
        df_test_agree.loc[i, 'error_gpt4'] = chat_completion.choices[0].message.content
 

In [558]:
# query v4:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt4']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt4'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt4'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.6379310344827587,
 0.6141490545050055,
 array([[ 7, 10,  2],
        [ 5, 28,  0],
        [ 0,  4,  2]], dtype=int64))

In [559]:
df_temp = df_test_agree[(df_test_agree['agreement_general']=='disagreement exists')|(df_test_agree['agreement_gpt4']=='disagreement exists')]
correct_l = []
is_correct_l = []
for i,row in df_temp.iterrows():
    if row['disagreement_areas']!=row['disagreement_areas']:
#         for a in row['disagreement_areas_gpt4'].split(', '):
#             if a != '':
#                 correct_l.append(False)
#                 is_correct_l.append(True)
        pass
    else:
        for a in row['disagreement_areas'].replace('Future Policy Stance', 'Future Policy Direction').split('; '):
            if a != '':
                correct_l.append(True)
                is_correct_l.append(a in row['disagreement_areas_gpt4'])
accuracy_score(correct_l, is_correct_l), confusion_matrix(correct_l, is_correct_l, labels=[True, False]) 

(0.2608695652173913,
 array([[ 6, 17],
        [ 0,  0]], dtype=int64))

4. other models

In [205]:
# version 5: gpt-4o-mini
for i, row in df_test_agree.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authority, determine whether the country's authority agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; possible areas include Current Policy Stance, Future Policy Direction, Monetary Policy Framework, Monetary Policy Operations, Central Bank Communication, Institutions, Policy Assessment, Economic Assessment, etc; if the authority mostly agree, leave the "disagreement_areas" field blank. Return a JSON dict without additional texts: \"agreement\", \"disagreement_areas\".''',},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])}
        ],
            model='gpt-4o-mini-2024-07-18', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace('\n',' '))
        df_test_agree.loc[i,'agreement_gpt5'] = result['agreement']
        df_test_agree.loc[i,'disagreement_areas_gpt5'] = result['disagreement_areas']
    except Exception:
        df_test_agree.loc[i, 'error_gpt5'] = chat_completion.choices[0].message.content
df_test_agree['error_gpt5'] = df_test_agree['error_gpt5'].apply(lambda x: json.loads(x.replace('```json','').replace('```','').replace('\n',' ')) if x==x else np.nan)
df_test_agree['agreement_gpt5'] = df_test_agree.apply(lambda x: x['error_gpt5']['agreement'] if x['agreement_gpt5']!=x['agreement_gpt5'] else x['agreement_gpt5'], axis=1)
df_test_agree['disagreement_areas_gpt5'] = df_test_agree.apply(lambda x: x['error_gpt5']['disagreement_areas'] if x['disagreement_areas_gpt5']!=x['disagreement_areas_gpt5'] else x['disagreement_areas_gpt5'], axis=1)

In [560]:
# query v5:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt5']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt5'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt5'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.6379310344827587,
 0.5933004926108374,
 array([[ 9, 10,  0],
        [ 5, 28,  0],
        [ 2,  4,  0]], dtype=int64))

In [561]:
df_temp = df_test_agree[(df_test_agree['agreement_general']=='disagreement exists')|(df_test_agree['agreement_gpt5']=='disagreement exists')]
correct_l = []
is_correct_l = []
for i,row in df_temp.iterrows():
    if row['disagreement_areas']!=row['disagreement_areas']:
#         for a in row['disagreement_areas_gpt5']:
#             if a != '':
#                 correct_l.append(False)
#                 is_correct_l.append(True)
        pass
    else:
        for a in row['disagreement_areas'].replace('Future Policy Stance', 'Future Policy Direction').split('; '):
            if a != '':
                correct_l.append(True)
                is_correct_l.append(row['disagreement_areas_gpt5']==row['disagreement_areas_gpt5'] and a in row['disagreement_areas_gpt5'])
accuracy_score(correct_l, is_correct_l), confusion_matrix(correct_l, is_correct_l, labels=[True, False]) 

(0.4782608695652174,
 array([[11, 12],
        [ 0,  0]], dtype=int64))

In [230]:
# version 6: gpt-3.5-turbo
for i, row in df_test_agree.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authority, determine whether the country's authority agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; possible areas include Current Policy Stance, Future Policy Direction, Monetary Policy Framework, Monetary Policy Operations, Central Bank Communication, Institutions, Policy Assessment, Economic Assessment, etc; if the authority mostly agree, leave the "disagreement_areas" field blank. Return a JSON dict without additional texts: \"agreement\", \"disagreement_areas\".''',},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])}
        ],
            model='gpt-3.5-turbo', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace('\n',' '))
        df_test_agree.loc[i,'agreement_gpt6'] = result['agreement']
        df_test_agree.loc[i,'disagreement_areas_gpt6'] = result['disagreement_areas']
    except Exception:
        df_test_agree.loc[i, 'error_gpt6'] = chat_completion.choices[0].message.content
df_test_agree['error_gpt6'] = df_test_agree['error_gpt6'].apply(lambda x: json.loads(x.replace('```json','').replace('```','').replace('\n',' ')) if x==x else np.nan)
df_test_agree['agreement_gpt6'] = df_test_agree.apply(lambda x: x['error_gpt6']['agreement'] if x['agreement_gpt6']!=x['agreement_gpt6'] else x['agreement_gpt6'], axis=1)
df_test_agree['disagreement_areas_gpt6'] = df_test_agree.apply(lambda x: x['error_gpt6']['disagreement_areas'] if x['disagreement_areas_gpt6']!=x['disagreement_areas_gpt6'] else x['disagreement_areas_gpt6'], axis=1)

In [562]:
# query v6:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt6']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt6'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt6'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.5862068965517241,
 0.5406896551724137,
 array([[ 7, 12,  0],
        [ 6, 27,  0],
        [ 3,  3,  0]], dtype=int64))

In [563]:
df_temp = df_test_agree[(df_test_agree['agreement_general']=='disagreement exists')|(df_test_agree['agreement_gpt6']=='disagreement exists')]
correct_l = []
is_correct_l = []
for i,row in df_temp.iterrows():
    if row['disagreement_areas']!=row['disagreement_areas']:
#         for a in row['disagreement_areas_gpt6']:
#             if a != '':
#                 correct_l.append(False)
#                 is_correct_l.append(True)
        pass
    else:
        for a in row['disagreement_areas'].replace('Future Policy Stance', 'Future Policy Direction').split('; '):
            if a != '':
                correct_l.append(True)
                is_correct_l.append(row['disagreement_areas_gpt6']==row['disagreement_areas_gpt6'] and a in row['disagreement_areas_gpt6'])
accuracy_score(correct_l, is_correct_l), confusion_matrix(correct_l, is_correct_l, labels=[True, False]) 

(0.2608695652173913,
 array([[ 6, 17],
        [ 0,  0]], dtype=int64))

4. fine-tuning

In [7]:
# monetary classification
df_train_agree['answer'] = df_train_agree.fillna('').apply(lambda x:( "{'agreement': '%s', 'disagreement_areas': %s}" % (x['agreement_general'], str(x['disagreement_areas'].replace('Future Policy Stance', 'Future Policy Direction').split('; ')))).replace("['']", "[]"), axis=1)
df_train_agree['messages'] = df_train_agree.apply(lambda row: [
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authorities, determine whether the country's authorities agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if the authorities do not directly express agreement/disagreement with IMF staff on monetary related issues, and either of the texts does not discuss monetary policy or they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy or if the authorities directly express agreement/disagreement to monetary related issues in IMF staff report, assign "disagreement exists" if the authorities disagree with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; possible areas include Current Policy Stance, Future Policy Direction, Monetary Policy Framework, Monetary Policy Operations, Central Bank Communication, Institutions, Policy Assessment, Economic Assessment, etc; if the authorities mostly agree, leave the "disagreement_areas" field blank. Return a JSON dict without additional texts: "agreement", "disagreement_areas".''',},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])},
                {   "role": "assistant",
                    "content":  row['answer']}
            ], axis=1)

df_train_agree[['messages']].to_json(directory+'output/finetuning/training_mon_agree.jsonl', orient='records', lines=True)


C:\Users\xtang2\AppData\Local\Temp\ipykernel_25196\2228181476.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_agree['answer'] = df_train_agree.fillna('').apply(lambda x:( "{'agreement': '%s', 'disagreement_areas': %s}" % (x['agreement_general'], str(x['disagreement_areas'].replace('Future Policy Stance', 'Future Policy Direction').split('; ')))).replace("['']", "[]"), axis=1)
C:\Users\xtang2\AppData\Local\Temp\ipykernel_25196\2228181476.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tra

In [584]:
# check data
import json
# import tiktoken # for token counting
import numpy as np
from collections import defaultdict

# Load the dataset
with open(directory+'output/finetuning/training_mon_agree.jsonl', 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

# Initial dataset stats
print("Num examples:", len(dataset))
print("First example:")
for message in dataset[0]["messages"]:
    print(message)

Num examples: 231
First example:
{'role': 'system', 'content': 'You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country\'s authority, determine whether the country\'s authority agree or disagree with IMF staff on issues related to the country\'s monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; possible areas include Current Policy Stance, Future Policy Direction, Monetary Policy Framework, Monetary Policy Operations, Central Bank Communication, Ins

In [585]:
# Format error checks
format_errors = defaultdict(int)

for ex in dataset:
    if not isinstance(ex, dict):
        format_errors["data_type"] += 1
        continue
        
    messages = ex.get("messages", None)
    if not messages:
        format_errors["missing_messages_list"] += 1
        continue
        
    for message in messages:
        if "role" not in message or "content" not in message:
            format_errors["message_missing_key"] += 1
        
        if any(k not in ("role", "content", "name", "function_call", "weight") for k in message):
            format_errors["message_unrecognized_key"] += 1
        
        if message.get("role", None) not in ("system", "user", "assistant", "function"):
            format_errors["unrecognized_role"] += 1
            
        content = message.get("content", None)
        function_call = message.get("function_call", None)
        
        if (not content and not function_call) or not isinstance(content, str):
            format_errors["missing_content"] += 1
    
    if not any(message.get("role", None) == "assistant" for message in messages):
        format_errors["example_missing_assistant_message"] += 1

if format_errors:
    print("Found errors:")
    for k, v in format_errors.items():
        print(f"{k}: {v}")
else:
    print("No errors found")

No errors found


In [586]:
# create finetuning job
file = client.files.create(
    file=open(directory+"output/finetuning/training_mon_agree.jsonl", "rb"),
    purpose="fine-tune"
)
print(file.id)

file-xEzLx1rBfImWUGbjx3KFl9Z3


In [587]:
client.fine_tuning.jobs.create(
    training_file=file.id,
    model="gpt-4o-mini-2024-07-18"
)

FineTuningJob(id='ftjob-RMPtQxZeeFQJX7XibIjvNkRt', created_at=1723507026, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-3AqEuQZh0o1lNQSUihUxYYSd', result_files=[], seed=226009844, status='validating_files', trained_tokens=None, training_file='file-xEzLx1rBfImWUGbjx3KFl9Z3', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)

In [399]:
# # Cancel a job
# client.fine_tuning.jobs.cancel("ftjob-dO9SinUakBn5OeagOU0hBpq1")

In [593]:
# List up to 10 events from a fine-tuning job
client.fine_tuning.jobs.list_events(fine_tuning_job_id="ftjob-RMPtQxZeeFQJX7XibIjvNkRt", limit=10)

SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-RjZWmK292p3e1Ae633hdU6vI', created_at=1723509016, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-Lx7GsUTYVCl7RhZJYL8W8fzI', created_at=1723509011, level='info', message='Usage policy evaluations completed, model is now enabled for sampling.', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-I3RrTBFVJ23tT1vB46fVPILL', created_at=1723508824, level='info', message='Evaluating model against our usage policies before enabling.', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-Ry94ybB6WfDLKzIpOeqyTw9e', created_at=1723508824, level='info', message='New fine-tuned model created: ft:gpt-4o-mini-2024-07-18:personal::9vZfLqyV', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-vxeHsqj5eoJaN2yrST4WO70Q', cre

In [594]:
# model_id = 'ft:gpt-4o-mini-2024-07-18:personal::9vUSWBSo'
# model_id = 'ft:gpt-3.5-turbo-0125:personal::9vW0uYV2'
# model_id = 'ft:gpt-4o-mini-2024-07-18:personal::9vWZOjh6'
model_id = 'ft:gpt-4o-mini-2024-07-18:personal::9vZfLqyV'

for i,row in df_test_agree.iterrows():
#     chat_completion = client.chat.completions.create(
#         messages=[
#                     {   "role": "user",
#                     "content":  '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authority, determine whether the country's authority agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; possible areas include Current Policy Stance, Future Policy Direction, Monetary Policy Framework, Monetary Policy Operations, Central Bank Communication, Institutions, Policy Assessment, Economic Assessment, etc; if the authority mostly agree, leave the "disagreement_areas" field blank. Provide reasoning for your answer. Return a JSON dict without additional texts: \"agreement\", \"disagreement_areas\", \"reason\". Please do not skip the reasoning part, and analyze your answer step by step.\n\nCountry: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])},
#         ],
#         model=model_id,
#     )
    chat_completion = client.chat.completions.create(
        messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts written by IMF staff and a country's authority, determine whether the country's authority agree or disagree with IMF staff on issues related to the country's monetary policy and assign a value to the "agreement" field": if either of the texts does not discuss monetary policy or if they discuss entirely different aspects of monetary policy, assign "irrelevant"; if the two texts discuss common aspect(s) of monetary policy, assign "disagreement exists" if the authority disagrees with IMF staff on any monetary policy issues, and "mostly agree" if no disagreement exists. If disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; possible areas include Current Policy Stance, Future Policy Direction, Monetary Policy Framework, Monetary Policy Operations, Central Bank Communication, Institutions, Policy Assessment, Economic Assessment, etc; if the authority mostly agree, leave the "disagreement_areas" field blank. Return a JSON dict without additional texts: \"agreement\", \"disagreement_areas\".''',},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])},
        ],
        model=model_id,
    )

    try:
        result = json.loads(chat_completion.choices[0].message.content.replace("'", '"').replace('```json','').replace('```','').replace('\n',' '))
        df_test_agree.loc[i,'agreement_gpt_ft4'] = result['agreement']
        df_test_agree.loc[i,'disagreement_areas_gpt_ft4'] = result['disagreement_areas']
#         df_test_agree.loc[i,'reason_gpt_ft2'] = result['reason']
    except Exception:
        df_test_agree.loc[i, 'error_gpt_ft4'] = chat_completion.choices[0].message.content
        
df_test_agree['error_gpt_ft4'] = df_test_agree['error_gpt_ft4'].apply(lambda x: json.loads(x.replace("'", '"').replace('s" ', "s' ").replace('"s ', "'s ").replace('```json','').replace('```','').replace('\n',' ')) if x==x else np.nan)
df_test_agree['agreement_gpt_ft4'] = df_test_agree.apply(lambda x: x['error_gpt_ft4']['agreement'] if x['agreement_gpt_ft4']!=x['agreement_gpt_ft4'] else x['agreement_gpt_ft4'], axis=1)
df_test_agree['disagreement_areas_gpt_ft4'] = df_test_agree.apply(lambda x: x['error_gpt_ft4']['disagreement_areas'] if x['disagreement_areas_gpt_ft4']!=x['disagreement_areas_gpt_ft4'] else x['disagreement_areas_gpt_ft4'], axis=1)
# df_test_agree['reason_gpt_ft2'] = df_test_agree.apply(lambda x: x['error_gpt_ft2']['reason'] if x['reason_gpt_ft2']!=x['reason_gpt_ft2'] else x['reason_gpt_ft2'], axis=1)

In [564]:
# ft v1:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft1']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft1'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft1'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.6551724137931034,
 0.6581566824680786,
 array([[12,  7,  0],
        [10, 22,  1],
        [ 0,  2,  4]], dtype=int64))

In [567]:
df_temp = df_test_agree[(df_test_agree['agreement_general']=='disagreement exists')|(df_test_agree['agreement_gpt_ft1']=='disagreement exists')]
correct_l = []
is_correct_l = []
for i,row in df_temp.iterrows():
    if row['disagreement_areas']!=row['disagreement_areas']:
#         for a in row['disagreement_areas_gpt_ft1'].split('; '):
#             if a != '':
#                 correct_l.append(False)
#                 is_correct_l.append(True)
        pass
    else:
        for a in row['disagreement_areas'].split('; '):
            if a != '':
                correct_l.append(True)
                is_correct_l.append(row['disagreement_areas_gpt_ft1']==row['disagreement_areas_gpt_ft1'] and a in row['disagreement_areas_gpt_ft1'])
accuracy_score(correct_l, is_correct_l), confusion_matrix(correct_l, is_correct_l, labels=[True, False]) 

(0.4782608695652174,
 array([[11, 12],
        [ 0,  0]], dtype=int64))

In [565]:
# ft v2:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft2']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft2'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft2'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.5689655172413793,
 0.5592278611524802,
 array([[11,  8,  0],
        [12, 21,  0],
        [ 1,  4,  1]], dtype=int64))

In [568]:
df_temp = df_test_agree[(df_test_agree['agreement_general']=='disagreement exists')|(df_test_agree['agreement_gpt_ft2']=='disagreement exists')]
correct_l = []
is_correct_l = []
for i,row in df_temp.iterrows():
    if row['disagreement_areas']!=row['disagreement_areas']:
#         for a in row['disagreement_areas_gpt_ft2'].split('; '):
#             if a != '':
#                 correct_l.append(False)
#                 is_correct_l.append(True)
        pass
    else:
        for a in row['disagreement_areas'].split('; '):
            if a != '':
                correct_l.append(True)
                is_correct_l.append(row['disagreement_areas_gpt_ft2']==row['disagreement_areas_gpt_ft2'] and a in row['disagreement_areas_gpt_ft2'])
accuracy_score(correct_l, is_correct_l), confusion_matrix(correct_l, is_correct_l, labels=[True, False]) 

(0.43478260869565216,
 array([[10, 13],
        [ 0,  0]], dtype=int64))

In [566]:
# ft v3:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft3']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft3'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft3'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.6896551724137931,
 0.6789029194323026,
 array([[11,  8,  0],
        [ 5, 27,  1],
        [ 1,  3,  2]], dtype=int64))

In [569]:
df_temp = df_test_agree[(df_test_agree['agreement_general']=='disagreement exists')|(df_test_agree['agreement_gpt_ft3']=='disagreement exists')]
correct_l = []
is_correct_l = []
for i,row in df_temp.iterrows():
    if type(row['disagreement_areas_gpt_ft3'])==str:
        row['disagreement_areas_gpt_ft3'] = [row['disagreement_areas_gpt_ft3']]
    if row['disagreement_areas']!=row['disagreement_areas']:
#         for a in row['disagreement_areas_gpt_ft3']:
#             if a != '':
#                 correct_l.append(False)
#                 is_correct_l.append(True)
        pass
    else:
        for a in row['disagreement_areas'].replace('Future Policy Stance', 'Future Policy Direction').split('; '):
            if a != '':
                correct_l.append(True)
                is_correct_l.append(row['disagreement_areas_gpt_ft3']==row['disagreement_areas_gpt_ft3'] and a in row['disagreement_areas_gpt_ft3'])
accuracy_score(correct_l, is_correct_l), confusion_matrix(correct_l, is_correct_l, labels=[True, False]) 

(0.5217391304347826,
 array([[12, 11],
        [ 0,  0]], dtype=int64))

In [595]:
# ft v4:
accuracy_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft4']), f1_score(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft4'], average='weighted'), confusion_matrix(df_test_agree['agreement_general'], df_test_agree['agreement_gpt_ft4'], labels=['disagreement exists', 'mostly agree', 'irrelevant'])

(0.6896551724137931,
 0.6805959613245487,
 array([[ 9, 10,  0],
        [ 3, 27,  3],
        [ 0,  2,  4]], dtype=int64))

In [596]:
df_temp = df_test_agree[(df_test_agree['agreement_general']=='disagreement exists')|(df_test_agree['agreement_gpt_ft4']=='disagreement exists')]
correct_l = []
is_correct_l = []
for i,row in df_temp.iterrows():
    if type(row['disagreement_areas_gpt_ft4'])==str:
        row['disagreement_areas_gpt_ft4'] = [row['disagreement_areas_gpt_ft4']]
    if row['disagreement_areas']!=row['disagreement_areas']:
#         for a in row['disagreement_areas_gpt_ft4']:
#             if a != '':
#                 correct_l.append(False)
#                 is_correct_l.append(True)
        pass
    else:
        for a in row['disagreement_areas'].replace('Future Policy Stance', 'Future Policy Direction').split('; '):
            if a != '':
                correct_l.append(True)
                is_correct_l.append(row['disagreement_areas_gpt_ft4']==row['disagreement_areas_gpt_ft4'] and a in row['disagreement_areas_gpt_ft4'])
accuracy_score(correct_l, is_correct_l), confusion_matrix(correct_l, is_correct_l, labels=[True, False]) 

(0.34782608695652173,
 array([[ 8, 15],
        [ 0,  0]], dtype=int64))

In [598]:
# save results
df_test_agree.drop('error_gpt_ft', axis=1).to_excel(directory+'output/finetuning/df_test_agree_mon.xlsx', index=False)